# 03 · Splits, storm events, and physics/persistence baselines
Test (2021–2025) and extra (2026) are **locked**: nothing below evaluates on them. All numbers are on `val` (2016–2018).

In [ ]:
import os, sys, json, pathlib, warnings
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "stormtime").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "src" / "stormtime").exists(), "Run inside the cloned repo (see README)"
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from stormtime import config as C, data as D, features as F, events as E, splits as S, baselines as B, metrics as M
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25, "axes.spines.top": False, "axes.spines.right": False})
RAW, CLEAN, FIG, RES = ROOT/"data/raw", ROOT/"data/clean", ROOT/"figures", ROOT/"results"
for p in (RAW, CLEAN, FIG, RES): p.mkdir(parents=True, exist_ok=True)
print("repo:", ROOT)

In [ ]:
df = pd.read_parquet(CLEAN / "omni_hourly_clean.parquet")
df = df.astype({c: "float64" for c in df.columns if str(df[c].dtype) == "float32"})
t = df.index
dst, P, vbs = df["dst"].to_numpy(), df["pressure"].to_numpy(), df["vbs"].to_numpy()
samples = S.make_sample_index(t)
rows = []
for name in C.SPLIT_ORDER:
    s, e = C.SPLITS[name]
    nh = int(((t >= pd.Timestamp(s)) & (t <= pd.Timestamp(e))).sum())
    rows.append({"split": name, "from": s[:10], "to": e[:10], "hours": nh, "origins_after_purge": int((samples.split == name).sum())})
pd.DataFrame(rows).set_index("split")

In [ ]:
try:
    S.get_split(samples, "test")
except PermissionError as e:
    print("guard works ->", e)

## 1 · Storm events by split (train/val/calib only)
Event rule: contiguous hours with Dst ≤ threshold, merged if separated by < 24 h.

In [ ]:
dst_nan = dst.copy()
rows = []
for thr in (-50, -100, -150, -250):
    for name in ("train", "val", "calib"):
        s, e = C.SPLITS[name]; m = (t >= pd.Timestamp(s)) & (t <= pd.Timestamp(e))
        ev = E.find_events(np.where(m, dst_nan, np.nan), thr)
        rows.append({"threshold": thr, "split": name, "n_events": len(ev), "deepest_nT": ev.min_dst.min() if len(ev) else np.nan})
evt = pd.DataFrame(rows).pivot(index="threshold", columns="split", values=["n_events", "deepest_nT"]); evt

In [ ]:
# Suggest an extreme-storm threshold from TRAIN counts only; override by hand if you disagree.
n_train = {thr: int(evt.loc[thr, ("n_events", "train")]) for thr in (-100, -150, -250)}
EXTREME_THRESHOLD = -150 if n_train[-150] >= 8 else -100
print("train event counts:", n_train, "-> EXTREME_THRESHOLD =", EXTREME_THRESHOLD)
json.dump({"extreme_threshold": EXTREME_THRESHOLD, "train_event_counts": n_train,
           "rule": "Dst<=thr, merge gap <24h, buffer +/-72h", "locked_before_model_results": True},
          open(RES / "protocol_choices.json", "w"), indent=1)

In [ ]:
s, e = C.SPLITS["train"]; mtr = (t >= pd.Timestamp(s)) & (t <= pd.Timestamp(e))
ev_tr = E.find_events(np.where(mtr, dst, np.nan), EXTREME_THRESHOLD)
ev_tr["min_time"] = t[ev_tr.min_idx.astype(int)]
excl = E.event_mask(len(t), ev_tr, C.EVENT_BUFFER_H)
print(f"Held-out-event experiment would remove {int((excl & mtr).sum())} of {int(mtr.sum())} train hours ({(excl & mtr).sum()/mtr.sum():.1%}) from {len(ev_tr)} events")
ev_tr.sort_values("min_dst").head(10)[["min_time", "min_dst", "span_h"]]

## 2 · Baselines on `val`
**persistence** · **climatology** · **Burton-fit** (constants fitted on train, exact exponential update) · **O'Brien–McPherron 2000 published forms** (constants VERIFY in `config.py`). *Oracle* rows are diagnostics, not forecasts.

In [ ]:
dstar = B.dst_to_star(dst, P)
tr_i = samples.loc[samples.split == "train", "i"].to_numpy()
pairs = tr_i[tr_i + 1 < len(t)]
fit = B.fit_burton(dstar, vbs, pairs)
print("Burton fit (train):", {k: round(v, 3) if isinstance(v, float) else v for k, v in fit.items()})
json.dump(fit, open(RES / "burton_fit.json", "w"), indent=1)
clim = float(np.nanmean(dst[mtr]))

In [ ]:
va = samples.loc[samples.split == "val", "i"].to_numpy()
all_rows, preds_store = [], {}
for h in C.HORIZONS:
    ok = B.common_eval_mask(dst, P, vbs, va, h)
    o = va[ok]; y = dst[o + h]
    preds = {"persistence": B.forecast_persistence(dst, o, h),
             "climatology": B.forecast_climatology(clim, len(o)),
             "burton_fit": B.forecast_physics(dst, P, vbs, o, h, "burton", fit),
             "om2000_published": B.forecast_physics(dst, P, vbs, o, h, "om2000"),
             "burton_fit [oracle, diagnostic]": B.forecast_physics(dst, P, vbs, o, h, "burton", fit, mode="oracle"),
             "om2000 [oracle, diagnostic]": B.forecast_physics(dst, P, vbs, o, h, "om2000", mode="oracle")}
    tab = M.table(y, preds); tab.insert(0, "horizon_h", h); all_rows.append(tab.reset_index())
    preds_store[h] = (o, y, preds)
    print(f"h={h}: {len(o)} of {len(va)} val origins scored ({len(o)/len(va):.1%})")
res = pd.concat(all_rows).set_index(["horizon_h", "model"]).round(3)
res.to_csv(RES / "baselines_val.csv"); res

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 3.4))
for m_ in res.index.get_level_values("model").unique():
    if m_ == "climatology": continue
    sub = res.xs(m_, level="model")
    ls = "--" if "oracle" in m_ else "-"
    axs[0].plot(sub.index, sub.RMSE, ls, marker="o", label=m_)
    axs[1].plot(sub.index, sub["skill_vs_persistence"], ls, marker="o")
    axs[2].plot(sub.index, sub.RMSE_storm, ls, marker="o")
axs[0].set_title("RMSE (nT), val"); axs[1].set_title("MSE skill vs persistence"); axs[2].set_title("RMSE in storm hours (Dst<=-50)")
for a in axs: a.set_xlabel("horizon (h)"); a.set_xticks(C.HORIZONS)
axs[1].axhline(0, color="k", lw=.7); axs[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIG / "03_baselines_val.png"); plt.show()

In [ ]:
# largest validation event at h=3, plotted at TARGET time
h = C.PRIMARY_HORIZON
ev_va = E.find_events(np.where((t >= pd.Timestamp(C.SPLITS["val"][0])) & (t <= pd.Timestamp(C.SPLITS["val"][1])), dst, np.nan), -100)
r = ev_va.sort_values("min_dst").iloc[0]
lo, hi = int(r.start_idx) - 72, int(r.end_idx) + 96
o_all = np.arange(lo, hi - h)
ok = B.common_eval_mask(dst, P, vbs, o_all, h); o = o_all[ok]
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(t[o + h], dst[o + h], "k", lw=2, label="observed Dst")
ax.plot(t[o + h], B.forecast_persistence(dst, o, h), label="persistence", alpha=.8)
ax.plot(t[o + h], B.forecast_physics(dst, P, vbs, o, h, "burton", fit), label="Burton-fit (forecast)")
ax.plot(t[o + h], B.forecast_physics(dst, P, vbs, o, h, "om2000"), label="OM2000 (forecast)")
ax.plot(t[o + h], B.forecast_physics(dst, P, vbs, o, h, "burton", fit, mode="oracle"), "--", label="Burton-fit [oracle, diagnostic]")
ax.set_title(f"Largest val storm, {h}-h forecasts (min {r.min_dst:.0f} nT at {t[int(r.min_idx)]:%Y-%m-%d %H:%M})"); ax.set_ylabel("Dst (nT)"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIG / "03_val_storm_h3.png"); plt.show()

In [ ]:
o, y, preds = preds_store[C.PRIMARY_HORIZON]
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for ax, name in zip(axs, ["persistence", "burton_fit"]):
    cm = M.confusion(y, preds[name]); ax.imshow(cm.values, cmap="Blues"); ax.grid(False)
    ax.set_xticks(range(4)); ax.set_xticklabels(C.CLASS_NAMES, rotation=30); ax.set_yticks(range(4)); ax.set_yticklabels(C.CLASS_NAMES)
    for (i, j), v in np.ndenumerate(cm.values): ax.text(j, i, v, ha="center", va="center", color="w" if v > cm.values.max() / 2 else "k", fontsize=8)
    ax.set_title(f"{name}, h=3, val"); ax.set_xlabel("predicted class"); ax.set_ylabel("observed class")
plt.tight_layout(); plt.savefig(FIG / "03_confusion_h3_baselines.png"); plt.show()
M.confusion(y, preds["burton_fit"])

In [ ]:
print("PRIMARY ENDPOINT (h=3, val): ")
print(res.xs(3, level="horizon_h")[["RMSE", "MAE", "r", "skill_vs_persistence", "RMSE_storm"]])

### Commit
```
git add results figures && git commit -m "Splits, events, baselines" && git push
```
Paste back: the split table, the event table, `Burton fit (train)`, and the `PRIMARY ENDPOINT` table. Those fix the numbers every neural model must beat.